# E3: Interactive ALFWorld Agent with CVX Causal Memory

Unlike E2 (static plan generation), this notebook runs an LLM agent **inside** the ALFWorld environment.
At each step, the agent observes the world, queries CVX for similar past states, and decides its next action.

### The Interactive Loop

```
1. env.reset() → observation, task
2. embed(observation) → query CVX for similar past states
3. CVX returns continuations: "from a similar state, agents did X next"
4. LLM(observation + continuations + admissible_actions) → action
5. env.step(action) → new observation
6. → goto 2 until done or max_steps
```

### Conditions

| Condition | Memory at Each Step |
|-----------|--------------------|
| **NoMemory** | Only current observation + admissible actions |
| **CVX-Causal** | Observation → search CVX → inject continuation from similar past states |

### Metric

**Task completion rate** (binary: did the agent solve the task within max_steps?)

In [1]:
import os, json, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv

# === CONFIG ===
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
MAX_STEPS = 30
N_EVAL = 30  # number of games to evaluate
TOP_K = 3

DATA_DIR = Path('../data/episodic')
embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL}')
print(f'Max steps: {MAX_STEPS}, Eval games: {N_EVAL}')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8014.94it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct
Max steps: 30, Eval games: 30


## 1. Load CVX Episodic Memory (from E2)

We reuse the index built in E2 from AgentInstruct expert trajectories.

In [2]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'alfworld_episodic.cvx')
META_PATH = str(DATA_DIR / 'alfworld_metadata.json')

assert os.path.exists(INDEX_PATH), f'Run E2 first to build {INDEX_PATH}'
index = cvx.TemporalIndex.load(INDEX_PATH)
with open(META_PATH) as f:
    episode_data = {int(k): v for k, v in json.load(f).items()}

print(f'CVX index: {len(index)} points ({len(episode_data)} episodes)')
print(f'Episode step range: {min(e["n_steps"] for e in episode_data.values())}-{max(e["n_steps"] for e in episode_data.values())}')

CVX index: 4542 points (336 episodes)
Episode step range: 3-35


## 2. Initialize ALFWorld Environment

In [3]:
config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0,
        'num_eval_games': 0,
    },
    'env': {
        'goal_desc_human_anns_prob': 0.0,
        'task_types': [1, 2, 3, 4, 5, 6],
        'domain_randomization': False,
        'expert_type': 'handcoded',
    },
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {
        'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
        'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2'),
    }
}

env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)

# Quick test
obs, info = env.reset()
task = obs[0].split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in obs[0] else 'unknown'
print(f'Test game: {task}')
print(f'Available actions: {len(info["admissible_commands"][0])}')
print(f'Total eval games: {len(env_wrapper.game_files)}')

Initializing AlfredTWEnv...


  0%|          | 0/341 [00:00<?, ?it/s]

 59%|█████▉    | 202/341 [00:00<00:00, 2016.28it/s]

100%|██████████| 341/341 [00:00<00:00, 1866.32it/s]

Overall we have 134 games in split=eval_out_of_distribution
Evaluating with 134 games


Test game: put a cool tomato in microwave.
Available actions: 28
Total eval games: 134


## 3. CVX Causal Retrieval (Step-by-Step)

The key difference from E2: the query is the **current observation**, not the task description.
This finds similar mid-episode states and returns what successful agents did next.

In [4]:
def retrieve_causal(observation, task, top_k=TOP_K):
    """Search CVX for similar past states and return continuations.
    
    Query = current observation (NOT task description).
    This finds similar mid-episode states in the expert memory.
    """
    # Embed the current state (observation + task for context)
    query_text = f'Task: {task} | Obs: {observation[:300]}'
    query_vec = embedder.encode(query_text).tolist()
    
    results = index.search(vector=query_vec, k=top_k * 20)
    
    seen = set()
    retrievals = []
    for node_id, ts, score in results:
        ep_idx = ts // 10000
        match_step = ts % 10000
        
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        seen.add(ep_idx)
        
        ep = episode_data[ep_idx]
        all_steps = ep['steps']
        
        # Get continuation: steps AFTER the match
        if match_step + 1 < len(all_steps):
            continuation = all_steps[match_step + 1:match_step + 6]  # next 5 steps max
        else:
            continuation = all_steps[-3:]  # fallback: last 3 steps
        
        retrievals.append({
            'episode_id': ep_idx,
            'match_step': match_step,
            'total_steps': len(all_steps),
            'task': ep['task'],
            'continuation': continuation,
            'similarity': score,
        })
        
        if len(retrievals) >= top_k:
            break
    
    return retrievals


def format_causal_context(retrievals):
    """Format retrieved continuations for the LLM prompt."""
    if not retrievals:
        return ''
    
    lines = ['Here is what successful agents did from similar situations:\n']
    for i, r in enumerate(retrievals, 1):
        lines.append(f'[Similar state from "{r["task"]}" (step {r["match_step"]}/{r["total_steps"]})]')
        lines.append('Next actions taken:')
        for step in r['continuation']:
            if isinstance(step, dict):
                lines.append(f'  > {step.get("action", "")}')
            else:
                lines.append(f'  > {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


# Test: retrieve from the current game state
retrievals = retrieve_causal(obs[0], task)
print(f'Query state: {obs[0][:100]}...\n')
print(f'Retrieved {len(retrievals)} continuations:')
for r in retrievals:
    print(f'  ep={r["episode_id"]}, step {r["match_step"]}/{r["total_steps"]}, '
          f'sim={r["similarity"]:.3f}, task="{r["task"][:40]}"')
    for s in r['continuation'][:2]:
        act = s.get('action', str(s)) if isinstance(s, dict) else str(s)
        print(f'    → {act[:60]}')

Query state: -= Welcome to TextWorld, ALFRED! =-

You are in the middle of a room. Looking quickly around you, yo...

Retrieved 3 continuations:
  ep=35, step 4/11, sim=0.334, task="cool some tomato and put it in microwave"
    → take tomato 1 from countertop 2
    → go to fridge 1
  ep=89, step 4/11, sim=0.357, task="cool some tomato and put it in microwave"
    → take tomato 2 from countertop 2
    → go to fridge 1
  ep=68, step 2/10, sim=0.371, task="put a cool tomato in microwave."
    → go to countertop 2
    → take tomato 1 from countertop 2


## 4. LLM Agent

The agent receives the observation, admissible actions, and optionally CVX causal context.
It must output exactly one of the admissible actions.

In [5]:
def agent_act(observation, admissible_actions, task, history, causal_context=''):
    """LLM agent: choose one admissible action."""
    system = (
        'You are an agent in a household environment. Your goal is to complete the task. '
        'You MUST choose exactly ONE action from the list of available actions. '
        'Output ONLY the action text, nothing else.'
    )
    
    user_parts = []
    user_parts.append(f'Task: {task}\n')
    
    if causal_context:
        user_parts.append(causal_context)
    
    if history:
        user_parts.append('Recent actions:')
        for h in history[-5:]:  # last 5 steps
            user_parts.append(f'  > {h["action"]} → {h["obs"][:80]}')
        user_parts.append('')
    
    user_parts.append(f'Current observation: {observation}\n')
    user_parts.append('Available actions:')
    for a in admissible_actions:
        user_parts.append(f'  - {a}')
    user_parts.append('\nChoose ONE action:')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=100,
    )
    
    chosen = response.choices[0].message.content.strip()
    
    # Fuzzy match: find the closest admissible action
    chosen_lower = chosen.lower().strip()
    for action in admissible_actions:
        if action.lower() == chosen_lower:
            return action
    
    # Partial match
    for action in admissible_actions:
        if action.lower() in chosen_lower or chosen_lower in action.lower():
            return action
    
    # Fallback: first action
    return admissible_actions[0]


def run_episode(env, use_memory=False, verbose=False):
    """Run one episode with or without CVX causal memory."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    
    history = []
    
    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        
        # CVX causal retrieval (if enabled)
        causal_context = ''
        if use_memory:
            retrievals = retrieve_causal(observation, task)
            causal_context = format_causal_context(retrievals)
        
        # Agent chooses action
        action = agent_act(observation, admissible, task, history, causal_context)
        
        if verbose:
            print(f'  Step {step+1}: {action}')
        
        # Execute
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        
        history.append({'action': action, 'obs': observation[:100]})
        
        if dones[0]:
            break
    
    won = info['won'][0]
    return {
        'task': task,
        'won': won,
        'steps': len(history),
        'history': history,
    }


# Test one episode with memory
print('=== Test Episode (with CVX-Causal) ===\n')
result = run_episode(env, use_memory=True, verbose=True)
print(f'\nTask: {result["task"]}')
print(f'Won: {result["won"]}, Steps: {result["steps"]}')

=== Test Episode (with CVX-Causal) ===



  Step 1: go to drawer 2


  Step 2: open drawer 2


  Step 3: close drawer 2


  Step 4: go to cabinet 1


...

  Step 26: go to cabinet 2


  Step 27: close cabinet 2


  Step 28: go to drawer 2


  Step 29: open drawer 2


  Step 30: examine drawer 2

Task: put a clean spatula in drawer.
Won: False, Steps: 30


## 5. Evaluation

Run N games for each condition and compare task completion rates.

In [6]:
conditions = {
    'NoMemory': False,
    'CVX-Causal': True,
}

all_results = {cond: [] for cond in conditions}
t0 = time.perf_counter()

for game_idx in range(N_EVAL):
    for cond, use_mem in conditions.items():
        result = run_episode(env, use_memory=use_mem)
        all_results[cond].append(result)
    
    if (game_idx + 1) % 5 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (game_idx + 1) * (N_EVAL - game_idx - 1)
        rates = {c: sum(r['won'] for r in res) / len(res) for c, res in all_results.items()}
        print(f'[{game_idx+1}/{N_EVAL}] {rates} (elapsed={elapsed:.0f}s, ETA={eta:.0f}s)')

print(f'\nDone in {time.perf_counter() - t0:.0f}s')

# Results
print('\n=== TASK COMPLETION RATE ===')
for cond, res in all_results.items():
    won = sum(r['won'] for r in res)
    total = len(res)
    mean_steps = np.mean([r['steps'] for r in res])
    print(f'{cond:>12}: {won}/{total} = {won/total:.1%}  (mean steps: {mean_steps:.1f})')

[5/30] {'NoMemory': 0.0, 'CVX-Causal': 0.0} (elapsed=222s, ETA=1109s)


[10/30] {'NoMemory': 0.0, 'CVX-Causal': 0.0} (elapsed=436s, ETA=871s)


[15/30] {'NoMemory': 0.0, 'CVX-Causal': 0.2} (elapsed=633s, ETA=633s)


[20/30] {'NoMemory': 0.05, 'CVX-Causal': 0.25} (elapsed=824s, ETA=412s)


[25/30] {'NoMemory': 0.04, 'CVX-Causal': 0.2} (elapsed=1036s, ETA=207s)


[30/30] {'NoMemory': 0.03333333333333333, 'CVX-Causal': 0.2} (elapsed=1234s, ETA=0s)

Done in 1234s

=== TASK COMPLETION RATE ===
    NoMemory: 1/30 = 3.3%  (mean steps: 29.3)
  CVX-Causal: 6/30 = 20.0%  (mean steps: 27.2)


In [7]:
import plotly.graph_objects as go

colors = {'NoMemory': '#95a5a6', 'CVX-Causal': '#2ecc71'}

# Completion rate bar chart
fig = go.Figure()
for cond, res in all_results.items():
    rate = sum(r['won'] for r in res) / len(res)
    fig.add_trace(go.Bar(
        x=[cond], y=[rate * 100],
        text=f'{rate:.1%}', textposition='outside',
        marker_color=colors.get(cond, '#333'),
        name=cond,
    ))

fig.update_layout(
    title=f'ALFWorld Task Completion Rate (n={N_EVAL}, max_steps={MAX_STEPS})',
    yaxis_title='Completion Rate (%)', yaxis=dict(range=[0, 100]),
    height=400, showlegend=False,
)
fig.show()

# Step distribution
fig2 = go.Figure()
for cond in conditions:
    steps = [r['steps'] for r in all_results[cond]]
    fig2.add_trace(go.Histogram(
        x=steps, name=cond, marker_color=colors.get(cond),
        opacity=0.7, nbinsx=15,
    ))

fig2.update_layout(
    title='Steps per Episode Distribution',
    xaxis_title='Steps', yaxis_title='Count',
    barmode='overlay', height=350,
)
fig2.show()

In [8]:
from scipy import stats

# McNemar's test
a_won = np.array([r['won'] for r in all_results['CVX-Causal']])
b_won = np.array([r['won'] for r in all_results['NoMemory']])

n_10 = np.sum(a_won & ~b_won)  # CVX-Causal only
n_01 = np.sum(~a_won & b_won)  # NoMemory only
n_11 = np.sum(a_won & b_won)
n_00 = np.sum(~a_won & ~b_won)

if n_10 + n_01 > 0:
    chi2 = (abs(n_10 - n_01) - 1) ** 2 / (n_10 + n_01)
    p_val = 1 - stats.chi2.cdf(chi2, df=1)
else:
    chi2, p_val = 0, 1.0

sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'

print('=== McNemar\'s Test: CVX-Causal vs NoMemory ===')
print(f'CVX-Causal only won: {n_10}')
print(f'NoMemory only won:   {n_01}')
print(f'Both won:            {n_11}')
print(f'Neither won:         {n_00}')
print(f'Net: {n_10 - n_01:+d} tasks, chi2={chi2:.2f}, p={p_val:.4f} {sig}')

# Summary
print(f'\n=== E3: INTERACTIVE ALFWORLD RESULTS ===')
print(f'Model: {LLM_MODEL}')
print(f'Memory: {len(episode_data)} expert episodes ({len(index)} vectors)')
print(f'Eval: {N_EVAL} games, max {MAX_STEPS} steps, eval_out_of_distribution')
print(f'\nNoMemory:   {sum(r["won"] for r in all_results["NoMemory"])}/{N_EVAL} = '
      f'{sum(r["won"] for r in all_results["NoMemory"])/N_EVAL:.1%}')
print(f'CVX-Causal: {sum(r["won"] for r in all_results["CVX-Causal"])}/{N_EVAL} = '
      f'{sum(r["won"] for r in all_results["CVX-Causal"])/N_EVAL:.1%}')
print(f'\nCVX features: search (step-by-step), episode encoding, temporal continuation')
print('This is the INTERACTIVE loop — query changes at each step based on env observation.')

=== McNemar's Test: CVX-Causal vs NoMemory ===
CVX-Causal only won: 5
NoMemory only won:   0
Both won:            1
Neither won:         24
Net: +5 tasks, chi2=3.20, p=0.0736 ns

=== E3: INTERACTIVE ALFWORLD RESULTS ===
Model: qwen2.5-coder:7b-instruct
Memory: 336 expert episodes (4542 vectors)
Eval: 30 games, max 30 steps, eval_out_of_distribution

NoMemory:   1/30 = 3.3%
CVX-Causal: 6/30 = 20.0%

CVX features: search (step-by-step), episode encoding, temporal continuation
This is the INTERACTIVE loop — query changes at each step based on env observation.
